In [1]:
# Conectar Colab con tu Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Importar las librerías básicas esenciales para predicción
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


Mounted at /content/drive


In [2]:
# 1. Ruta corregida según tu estructura real en español
ruta_real = '/content/drive/MyDrive/Data_Science_Projects/prediccion-precios-vivienda/data/raw/madrid_houses.csv'

# 2. Leer el archivo CSV real
import pandas as pd
df_viviendas = pd.read_csv(ruta_real)

# 3. Verificar cuántos datos reales tenemos y si faltan valores
print("📊 Conteo de filas y columnas reales:")
print(df_viviendas.shape)

print("\n🔍 Datos faltantes reales por cada columna:")
print(df_viviendas.isnull().sum())

📊 Conteo de filas y columnas reales:
(15975, 9)

🔍 Datos faltantes reales por cada columna:
price             0
house_type        0
house_type_2    469
rooms             0
m2                0
elevator          0
garage            0
neighborhood      0
district          0
dtype: int64


In [3]:
# 1. Limpieza: Rellenar los 469 datos faltantes de 'house_type_2'
df_viviendas['house_type_2'] = df_viviendas['house_type_2'].fillna('Desconocido')

# 2. Verificar que ya NO quedan datos faltantes en ninguna columna
print("🧹 Datos faltantes después de la limpieza:")
print(df_viviendas.isnull().sum())

# 3. Análisis de Negocio: Ver el precio promedio, mínimo y máximo en Madrid
print("\n📊 Resumen matemático de los inmuebles:")
display(df_viviendas[['price', 'm2', 'rooms']].describe())

# 4. Métrica Estrella: Calcular el precio promedio por metro cuadrado (€/m²)
# Esto le fascina a los reclutadores porque demuestra visión de negocio
precio_medio_m2 = (df_viviendas['price'] / df_viviendas['m2']).mean()
print(f"\n💰 El precio promedio por metro cuadrado en Madrid es de: {precio_medio_m2:.2f} €/m²")

🧹 Datos faltantes después de la limpieza:
price           0
house_type      0
house_type_2    0
rooms           0
m2              0
elevator        0
garage          0
neighborhood    0
district        0
dtype: int64

📊 Resumen matemático de los inmuebles:


,price,m2,rooms
count,1.597500e+04,15975.000000,15975.000000
mean,6.242327e+05,124.807398,2.847762
std,7.709074e+05,101.705064,1.360926
min,7.250000e+02,1.000000,1.000000
25%,1.950000e+05,66.000000,2.000000
50%,3.599730e+05,93.000000,3.000000
75%,7.490000e+05,142.000000,3.000000
max,1.395000e+07,989.000000,41.000000



💰 El precio promedio por metro cuadrado en Madrid es de: 12464.18 €/m²


In [4]:
# 1. Filtrar la tabla para quedarnos solo con casas reales y lógicas
# - Que midan más de 20 metros cuadrados y menos de 500 m²
# - Que cuesten más de 50,000 euros (eliminamos alquileres colados)
# - Que tengan menos de 7 habitaciones
df_filtrado = df_viviendas[
    (df_viviendas['m2'] >= 20) & (df_viviendas['m2'] <= 500) &
    (df_viviendas['price'] >= 50000) &
    (df_viviendas['rooms'] <= 7)
]

print(f"✂️ Filtro de Outliers: Eliminamos {df_viviendas.shape[0] - df_filtrado.shape[0]} registros extraños.")
print(f"🏠 Nos quedan {df_filtrado.shape[0]} casas perfectas para entrenar el modelo.")

# 2. Recalcular nuestra métrica estrella con los datos limpios
nuevo_precio_medio_m2 = (df_filtrado['price'] / df_filtrado['m2']).mean()
print(f"\n💰 El NUEVO precio promedio real en Madrid es de: {nuevo_precio_medio_m2:.2f} €/m²")

# 3. Guardar el progreso: Actualizamos nuestra variable principal
df_viviendas = df_filtrado.copy()

✂️ Filtro de Outliers: Eliminamos 306 registros extraños.
🏠 Nos quedan 15669 casas perfectas para entrenar el modelo.

💰 El NUEVO precio promedio real en Madrid es de: 4390.51 €/m²


In [5]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score
import pandas as pd

# 1. Seleccionar las características (X) y lo que queremos predecir (y)
# Usaremos tamaño, habitaciones, ascensor, garaje y el distrito de Madrid
X = df_viviendas[['m2', 'rooms', 'elevator', 'garage', 'district']]
y = df_viviendas['price']

# 2. Convertir textos (como los nombres de los distritos) en números automáticamente
# La IA solo entiende de matemáticas, así que este truco convierte categorías en columnas numéricas
X = pd.get_dummies(X, drop_first=True)

# 3. Dividir los datos: 80% entrenamiento, 20% pruebas
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"🧠 Casas asignadas para estudiar (Entrenamiento): {X_train.shape[0]}")
print(f"🧪 Casas guardadas para el examen final (Pruebas): {X_test.shape[0]}")

# 4. Crear el algoritmo (Usaremos Random Forest: un bosque de árboles de decisión, el rey para datos tabulares)
modelo_ia = RandomForestRegressor(n_estimators=50, max_depth=12, random_state=42)

# 5. ¡A ENTRENAR! Aquí la IA empieza a calcular la relación entre m², zonas y precios
print("\n⏳ Entrenando el modelo predictivo... (esto puede tomar de 5 a 10 segundos)")
modelo_ia.fit(X_train, y_train)
print("✅ ¡Inteligencia Artificial entrenada con éxito!")

# 6. Poner a prueba el modelo con el examen de las casas sorpresa
predicciones = modelo_ia.predict(X_test)

# 7. Calcular las métricas de rendimiento profesionales
error_medio = mean_absolute_error(y_test, predicciones)
precision_r2 = r2_score(y_test, predicciones)

print("\n🎯 === EVALUACIÓN DEL PORTAFOLIO ===")
print(f"💵 Error Absoluto Medio (MAE): {error_medio:.2f} €")
print(f"📈 Precisión del Modelo (R² Score): {precision_r2 * 100:.2f}%")

🧠 Casas asignadas para estudiar (Entrenamiento): 12535
🧪 Casas guardadas para el examen final (Pruebas): 3134

⏳ Entrenando el modelo predictivo... (esto puede tomar de 5 a 10 segundos)
✅ ¡Inteligencia Artificial entrenada con éxito!

🎯 === EVALUACIÓN DEL PORTAFOLIO ===
💵 Error Absoluto Medio (MAE): 125237.52 €
📈 Precisión del Modelo (R² Score): 86.50%


In [6]:
# 1. Definir las características del apartamento que quieres cotizar
mi_apartamento_imaginario = {
    'm2': 90,              # Tamaño en metros cuadrados
    'rooms': 3,            # Número de habitaciones
    'elevator': True,      # ¿Tiene ascensor? (True/False)
    'garage': True,        # ¿Tiene garaje? (True/False)
    'district': 'Retiro'   # Elige un distrito (ej: 'Centro', 'Retiro', 'Salamanca', 'Vallecas')
}

# 2. Convertir tu apartamento al formato matemático que entiende la IA
df_nuevo = pd.DataFrame([mi_apartamento_imaginario])
df_nuevo_codificado = pd.get_dummies(df_nuevo)
df_nuevo_codificado = df_nuevo_codificado.reindex(columns=X.columns, fill_value=0)

# 3. ¡Hacer la predicción!
precio_predicho = modelo_ia.predict(df_nuevo_codificado)[0]

print("🔮 === PREDICCIÓN DE LA INTELIGENCIA ARTIFICIAL ===")
print(f"🏠 Para un piso de {mi_apartamento_imaginario['m2']}m², {mi_apartamento_imaginario['rooms']} hab, con ascensor y garaje en el distrito de {mi_apartamento_imaginario['district']}:")
print(f"💰 El precio de mercado estimado por tu modelo es: {precio_predicho:,.2f} €")

🔮 === PREDICCIÓN DE LA INTELIGENCIA ARTIFICIAL ===
🏠 Para un piso de 90m², 3 hab, con ascensor y garaje en el distrito de Retiro:
💰 El precio de mercado estimado por tu modelo es: 335,399.17 €
